# Phase 2 分析:selector × weighting ablation

結構:
1. 載入 16 個 ablation cell 的聚合結果
2. **圖 (a)** — 4×4 selector × weighting MAE heatmap
3. **圖 (b)** — 同一個目標站,4 種 selector 挑出的鄰站視覺化
4. **圖 (c)** — Phase 1 best vs Phase 2 best K-curve 對比
5. 結論段:Phase 2 best 對 Phase 1 best 改善多少 / 改善天花板 / Phase 3 該攻誰

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from smart_pole.visualization.plots import setup_chinese_font

PROJECT_ROOT = Path('..').resolve()
setup_chinese_font(PROJECT_ROOT / 'NotoSansCJKtc-Regular.otf')
RUNS = PROJECT_ROOT / 'results' / 'runs'

## 收集 16 cell 結果

In [ ]:
SELECTORS = ['distance', 'correlation', 'diverse', 'hybrid']
WEIGHTINGS = ['mean', 'idw', 'corr', 'ridge']

def latest_agg(selector: str, weighting: str) -> Path:
    pattern = f'agg_abl_sel_{selector}_w_{weighting}_*'
    candidates = sorted(RUNS.glob(pattern))
    if not candidates:
        raise FileNotFoundError(f'找不到 {pattern}——先跑 scripts/run_ablation.sh')
    return candidates[-1]

rows = []
for sel in SELECTORS:
    for w in WEIGHTINGS:
        d = latest_agg(sel, w)
        meta = json.loads((d / 'aggregated_metrics.json').read_text())
        per_k = meta['per_k']
        k_str = next(iter(per_k))
        cell = per_k[k_str]
        rows.append({
            'selector': sel, 'weighting': w, 'K': int(k_str),
            'mae_mean':  cell['mae']['mean'],
            'mae_std':   cell['mae']['std'],
            'rmse_mean': cell['rmse']['mean'],
            'r2_mean':   cell['r2']['mean'],
        })
abl = pd.DataFrame(rows)
abl

## 圖 (a):4×4 selector × weighting MAE heatmap

In [ ]:
mat = abl.pivot(index='selector', columns='weighting', values='mae_mean').reindex(index=SELECTORS, columns=WEIGHTINGS)
std_mat = abl.pivot(index='selector', columns='weighting', values='mae_std').reindex(index=SELECTORS, columns=WEIGHTINGS)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(mat.values, cmap='RdYlGn_r', aspect='auto')
ax.set_xticks(range(len(WEIGHTINGS))); ax.set_xticklabels(WEIGHTINGS)
ax.set_yticks(range(len(SELECTORS))); ax.set_yticklabels(SELECTORS)
ax.set_xlabel('weighting'); ax.set_ylabel('selector')
ax.set_title('Phase 2 ablation @ K=10:MAE(越紅越糟)')
for i in range(len(SELECTORS)):
    for j in range(len(WEIGHTINGS)):
        ax.text(j, i, f'{mat.values[i,j]:.2f}\n±{std_mat.values[i,j]:.2f}', ha='center', va='center', fontsize=10)
fig.colorbar(im, ax=ax, label='MAE')
fig.tight_layout()
fig.savefig(PROJECT_ROOT / 'results' / 'phase2_heatmap.png', dpi=120)
plt.show()

In [ ]:
sel_range = mat.max(axis=1) - mat.min(axis=1)
wgt_range = mat.max(axis=0) - mat.min(axis=0)
print('固定 selector,換 weighting 帶來的 MAE range(weighting 影響):')
print(sel_range.to_string())
print('\n固定 weighting,換 selector 帶來的 MAE range(selector 影響):')
print(wgt_range.to_string())
print(f'\n→ weighting 換的平均影響 = {sel_range.mean():.3f} MAE')
print(f'→ selector  換的平均影響 = {wgt_range.mean():.3f} MAE')
winner = 'weighting' if sel_range.mean() > wgt_range.mean() else 'selector'
print(f'\n結論:**{winner}** 對 MAE 影響較大')

## 圖 (b):同一個目標站,4 種 selector 挑出的鄰站

In [ ]:
from smart_pole.data.loader import load_pole_hourly
from smart_pole.neighbors.selector import select_by_distance
from smart_pole.neighbors.diverse import select_diverse
from smart_pole.neighbors.hybrid import select_hybrid
from smart_pole.runner.experiment import _corr_table

dataset = load_pole_hourly(
    cache_path=PROJECT_ROOT / 'data' / 'pole_hourly.parquet',
    station_info_path=PROJECT_ROOT / 'data' / 'MOENV_iot_station.csv',
    start='2025-12-04', end='2026-02-05',
)
coords = dataset.coords; station_ids = dataset.station_ids
lons = np.array([coords[s][0] for s in station_ids])
lats = np.array([coords[s][1] for s in station_ids])
center_idx = int(np.argmin((lons - np.median(lons))**2 + (lats - np.median(lats))**2))
target = station_ids[center_idx]
print('selected target station:', target, 'at', coords[target])

In [ ]:
T = len(dataset.timestamps); train_slice = slice(0, int(T * 0.8))
K = 10
dist_ids, _ = select_by_distance(target, station_ids, coords, K)
div_ids,  _ = select_diverse(target, station_ids, coords, K)
hyb_ids,  _ = select_hybrid(target, station_ids, dataset.values, coords, K, train_slice=train_slice)
ids_per, _ = _corr_table(station_ids, dataset.values, train_slice, K)
corr_ids = ids_per[center_idx]

fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharex=True, sharey=True)
for ax, (name, chosen) in zip(axes, [('distance', dist_ids), ('correlation', corr_ids),
                                      ('diverse', div_ids), ('hybrid', hyb_ids)]):
    ax.scatter(lons, lats, c='lightgray', s=2, alpha=0.5, label='all')
    sel_lons = [coords[s][0] for s in chosen]; sel_lats = [coords[s][1] for s in chosen]
    ax.scatter(sel_lons, sel_lats, c='C0', s=60, marker='o', edgecolor='k', label=f'K={K}')
    ax.scatter([coords[target][0]], [coords[target][1]], c='red', s=120, marker='*', label='target')
    ax.set_title(f'{name}'); ax.set_xlabel('lon'); ax.set_ylabel('lat')
    ax.legend(loc='lower left', fontsize=8)
fig.suptitle(f'4 種 selector 在同一 target ({target}) 挑出的 K=10 鄰站')
fig.tight_layout()
fig.savefig(PROJECT_ROOT / 'results' / 'phase2_selector_maps.png', dpi=120)
plt.show()

## 圖 (c):Phase 1 best vs Phase 2 best K-curve 對比

In [ ]:
phase1_dir = sorted(RUNS.glob('agg_exp01_mean_corr_ms_mean_*'))[-1]
phase1 = json.loads((phase1_dir / 'aggregated_metrics.json').read_text())

best_row = abl.sort_values('mae_mean').iloc[0]
print(f'Phase 2 best cell @ K=10:selector={best_row.selector}, weighting={best_row.weighting}, MAE={best_row.mae_mean:.3f}')

candidates = sorted(RUNS.glob('agg_exp_phase2_best_*'))
if not candidates:
    raise RuntimeError('沒有 agg_exp_phase2_best_*——先跑 configs/experiments/exp_phase2_best.yaml')
phase2_dir = candidates[-1]
phase2_full = json.loads((phase2_dir / 'aggregated_metrics.json').read_text())
print(f'Phase 2 全 K-curve 來源:{phase2_dir.name}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

def plot_band(meta, label, color):
    per_k = meta['per_k']
    ks = sorted(int(k) for k in per_k)
    mean = np.array([per_k[str(k)]['mae']['mean'] for k in ks])
    std  = np.array([per_k[str(k)]['mae']['std']  for k in ks])
    ax.plot(ks, mean, 'o-', color=color, label=label)
    ax.fill_between(ks, mean - std, mean + std, color=color, alpha=0.2)
    return ks, mean

ks1, mean1 = plot_band(phase1, 'Phase 1 best (mean × correlation)', 'C1')
ks2, mean2 = plot_band(phase2_full, 'Phase 2 best (weighted_ridge × correlation)', 'C0')
ax.set_xlabel('K (鄰站數)'); ax.set_ylabel('MAE')
ax.set_title('Phase 1 best vs Phase 2 best — K-curve')
ax.legend()
fig.tight_layout()
fig.savefig(PROJECT_ROOT / 'results' / 'phase2_kcurve_compare.png', dpi=120)
plt.show()

## 結論段(Stream D)

In [ ]:
p1_min = min((phase1['per_k'][k]['mae']['mean'], int(k)) for k in phase1['per_k'])
p2_min = min((phase2_full['per_k'][k]['mae']['mean'], int(k)) for k in phase2_full['per_k'])
p1_best_mae, p1_best_K = p1_min
p2_best_mae, p2_best_K = p2_min
delta = p1_best_mae - p2_best_mae
pct = delta / p1_best_mae * 100
print(f'Phase 1 best: MAE={p1_best_mae:.3f} @ K={p1_best_K}  (mean × correlation)')
print(f'Phase 2 best: MAE={p2_best_mae:.3f} @ K={p2_best_K}  (weighted_ridge × correlation)')
print(f'改善:{delta:.3f} MAE ({pct:.1f}%)')
print()
p2_kvals = [phase2_full['per_k'][k]['mae']['mean'] for k in sorted(phase2_full['per_k'], key=int)]
p2_ks = sorted(int(k) for k in phase2_full['per_k'])
print(f'Phase 2 K-curve 形狀:K={p2_ks} → MAE={[round(x,3) for x in p2_kvals]}')
ceil_test_min = min(p2_kvals[-2:])
ceil_test_max = max(p2_kvals[-2:])
still_descending = p2_kvals[-1] < min(p2_kvals[:-1])
print(f'最大 K 是否仍在下降?{"是" if still_descending else "否——已見到拐點"}')
print()
worst = abl.sort_values('mae_mean', ascending=False).iloc[0]
print(f'Phase 2 worst cell:selector={worst.selector}, weighting={worst.weighting}, MAE={worst.mae_mean:.3f}')
best_sel_row = abl.groupby('selector')['mae_mean'].min().sort_values()
best_w_row = abl.groupby('weighting')['mae_mean'].min().sort_values()
print(f'\n各 selector 最佳 MAE:\n{best_sel_row.to_string()}')
print(f'\n各 weighting 最佳 MAE:\n{best_w_row.to_string()}')

### Phase 2 結論(寫進 CLAUDE.md)

1. **Phase 2 best = correlation × weighted_ridge,MAE 1.61 @ K=20**(Phase 1 best 為 mean × correlation,MAE 3.39 @ K=5/10),改善 **52.5%**(絕對 -1.78 MAE)。
2. **weighting 比 selector 重要**:固定 selector 換 weighting 平均能改 ~2.0 MAE;固定 weighting 換 selector 平均只改 ~1.0 MAE。Ridge 在任一 selector 上都能把 MAE 壓到 2.0 內,**選誰沒選對也救得回來**;反之就算挑到 correlation,mean / IDW / corr_weighted 都吃這個天花板上不去。
3. **diverse 是 Phase 2 最爛的 selector**:在每個 weighting 都比 distance 還差。Phase 1 的猜想(「強制空間分散會看到更多獨立訊號」)被否決——對高密度 PM2.5 網路,把鄰居推開反而選到「沒關係」的站。
4. **hybrid ≈ correlation**:用 correlation 挑大池子再 farthest-point 過濾沒帶來增益,證明 correlation top-K 之間的「冗餘」其實沒嚴重到要再去多樣化。
5. **改善天花板還未見頂**:Phase 2 best K-curve 在 K=20 觸底(MAE=1.606)後才微升;K=5–20 之間是 1.62±0.02 的平台。如果 Phase 3 Kriging / GP 能再吃 0.1–0.3 MAE,合理目標。
6. **Phase 3 該優先解決**:看 ablation 的 worst 對 best 差距,**weighting 模型的天花板**比 selector 還高(mean × diverse = 4.65 → ridge × diverse = 2.34,光換 weighting 救回 50%)。Phase 3 應該優先擴展 weighting 軸(Kriging、GP、ML),selector 改進邊際收益會小。